## El primer gran proyecto -

### y uso de herramientas.

### Pero primero presentamos Pushover

Pushover es una herramienta ingeniosa para enviar notificaciones push a tu teléfono.

¡Es muy fácil de configurar e instalar!

Simplemente visita https://pushover.net/ y haz clic en 'Login or Signup' en la parte superior derecha para registrarte con una cuenta gratuita y crear tus claves de API.

Una vez que te hayas registrado, en la pantalla de inicio, haz clic en "Create an Application/API Token", asígnale cualquier nombre (como Agents) y haz clic en Create Application.

Luego agrega 2 líneas a tu archivo .env:

PUSHOVER_USER=pon la clave que está en la parte superior derecha de tu pantalla de inicio de Pushover y que probablemente comienza con una u
PUSHOVER_TOKEN=pon la clave que aparece al hacer clic en tu nueva aplicación llamada Agents (o como la hayas llamado) y que probablemente comienza con una a

Recuerda guardar tu archivo .env y ejecutar load_dotenv(override=True) después de guardarlo, para establecer tus variables de entorno.

Finalmente, haz clic en "Add Phone, Tablet or Desktop" para instalarlo en tu teléfono.

In [1]:
# imports

from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr

In [2]:
# el inicio usual
## Le dice a la librería que viaje a Groq
load_dotenv(override=True)

openai = OpenAI(
    base_url="https://api.groq.com/openai/v1",  
    api_key=os.getenv("GROQ_API_KEY")
)

In [3]:
# para pushover

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with u
Pushover token found and starts with a


In [ ]:
# funcion para realizar un mensaje
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [ ]:
# llamo a la funcion y  envio un mensaje
push("Hola chico!!")

Push: Hola chico!!


### Introducción al ejercicio: El uso de Herramientas (Tool Calling)
En estas celdas vas a preparar a tu Agente para que no solo hable, sino que actúe. En el mundo de la IA, esto se conoce como Tool Calling (Uso de Herramientas) o Function Calling.

Hasta ahora, si un reclutador te dejaba su correo o hacía una pregunta que no sabías, el modelo solo podía responder con texto. Con estas 4 celdas estás creando el mecanismo para que el modelo pueda:

Detectar la intención del usuario (ej: "Quiero contactar con Jose" o "¿Jose sabe programar en COBOL?" —algo que no está en su LinkedIn—).

Decidir usar una herramienta específica para solucionar esa situación.

Enviar una alerta push a tu móvil en tiempo real notificándote el evento.

Para lograr esto, el proceso se divide en dos partes por cada acción: La función real en Python (lo que ejecuta la acción) y El esquema JSON (el "manual de instrucciones" que le enviamos a la IA para que entienda cuándo y cómo usar la función).

In [ ]:
# Función para registrar datos de contacto
# "Registrando el interés de {name} con correo electrónico {email} y notas {notes}")
def record_user_details(email, name="Name not provided", notes="not provided"):
    # Envía una notificación push a tu móvil con los datos capturados por el agente
    push(f"Recording interest from {name} with email {email} and notes {notes}")
    # Devuelve una respuesta estructurada para que el modelo sepa que la acción fue exitosa
    return {"recorded": "ok"}

In [6]:
# Función para registrar preguntas no respondidas

def record_unknown_question(question):
    # Te avisa al móvil en tiempo real que alguien ha preguntado algo que tu clon no sabe
    push(f"Recording {question} asked that I couldn't answer")
    # Devuelve confirmación al modelo de que la pregunta fantasma ha sido registrada
    return {"recorded": "ok"}

In [7]:
# Manual JSON para la herramienta de contacto
#Este JSON es la descripción técnica que leerá el Modelo (Groq/Gemini) para saber que 
# esta herramienta existe y qué datos obligatorios necesita pedirle al usuario.


record_user_details_json = {
    "name": "record_user_details",  # El nombre exacto de la función de Python a la que mapea
    "description": "Utiliza esta herramienta para registrar qu un usuario està interesado en estar en contacto y proporcionò una direciòn de correo electrònico",
    "parameters": {
        "type": "object",  # Define que los argumentos se pasarán como un objeto/diccionario
        "properties": {    # Especifica cada uno de los parámetros que la función puede recibir
            "email": {
                "type": "string",
                "description": "La direcciòn de correo electrònico de este usuario"
            },
            "name": {
                "type": "string",
                "description": "El nombre del usuario, si lo proporciona"
            }
            ,
            "notes": {
                "type": "string",
                "description": "Cualquier informaciòn adicional sobre la conversaciòn que merezca ser registrada para dar contexto"
            }
        },
        "required": ["email"],          # ¡Crucial! El email es obligatorio; el nombre y las notas son opcionales
        "additionalProperties": False   # Prohíbe a la IA inventarse parámetros que no estén definidos aquí
    }
}

In [8]:
# Manual JSON para la herramienta de preguntas desconocidas
#Este JSON le explica a la IA bajo qué condiciones estrictas debe activar la herramienta de registro de dudas.


record_unknown_question_json = {
    "name": "record_unknown_question",   # El nombre exacto de la función de Python a la que mapea
    "description": "Siempre use esta herramienta para registrar cualquier pregunta que no se pueda responder, ya que no sabia la respuesta",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "La pregunta que no se pudo responder"
            },
        },
        "required": ["question"],        # La pregunta que originó el fallo de conocimiento es obligatoria
        "additionalProperties": False     # No se permiten parámetros extra
    }
}

In [9]:
# empaquetar
# Creamos la lista oficial de herramientas que le pasaremos al modelo en la API
tools = [{"type": "function", "function": record_user_details_json}, # Empaquetamos la primera herramienta (Registrar Datos de Contacto)
        {"type": "function", "function": record_unknown_question_json}]  # Empaquetamos la segunda herramienta (Registrar Pregunta Desconocida)

In [10]:
tools

[{'type': 'function',
  'function': {'name': 'record_user_details',
   'description': 'Utiliza esta herramienta para registrar qu un usuario està interesado en estar en contacto y proporcionò una direciòn de correo electrònico',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': 'La direcciòn de correo electrònico de este usuario'},
     'name': {'type': 'string',
      'description': 'El nombre del usuario, si lo proporciona'},
     'notes': {'type': 'string',
      'description': 'Cualquier informaciòn adicional sobre la conversaciòn que merezca ser registrada para dar contexto'}},
    'required': ['email'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'record_unknown_question',
   'description': 'Siempre use esta herramienta para registrar cualquier pregunta que no se pueda responder, ya que no sabia la respuesta',
   'parameters': {'type': 'object',
    'properties': {'question': {'type': 'st

Explicación: ¿Cómo funciona handle_tool_calls?
Cuando el modelo de IA decide que necesita usar una herramienta, 
él no la ejecuta directamente. El modelo simplemente te devuelve una orden
 en su respuesta que dice: "Quiero que ejecutes la función X con los argumentos Y".

La función handle_tool_calls es el "motor de ejecución" en tu código Python. Su trabajo es:

1-Leer la lista de herramientas que la IA ha pedido activar (tool_calls).

2-Extraer el nombre de la función y los argumentos que la IA ha calculado (vía json.loads).

3-Buscar y ejecutar la función real en tu script pasándole esos argumentos (arguments).

4-Empaquetar la respuesta de la función en el formato oficial de la API ({"role": "tool", "content": ...})
 para devolvérsela a la IA y que sepa que la tarea se completó.

In [ ]:
# Esta funciòn puede tomar una lista de llamadas a herramientas y ejcutarlas. Este es el IF statement!!
# NO LA EJECUTO

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)

        # EL GRAN IF!!!

        if tool_name == "record_user_details":
            result = record_user_details(**arguments)
        elif tool_name == "record_unknown_question":
            result = record_unknown_question(**arguments)

        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

Celda de Prueba (Globals)
Al ejecutar esta celda, estás probando el truco de usar globals(). 
En Python, globals() es un diccionario que contiene todo lo que está vivo en la memoria de tu script. 
Escribir globals()["record_unknown_question"](...) es exactamente lo mismo que llamar a la función de manera normal,
pero buscándola por su nombre en texto.

In [11]:
globals()["record_unknown_question"]("this is a really hard question")

Push: Recording this is a really hard question asked that I couldn't answer


{'recorded': 'ok'}

Celda de la Función handle_tool_calls (Versión Elegante)
En lugar de usar un if/elif gigante que tendrías que modificar cada vez que añadas una herramienta nueva,
esta versión elegante usa globals().get(tool_name). Va al diccionario de la memoria de Python, busca la función
que coincide con el texto que envió la IA y la ejecuta dinámicamente en una sola línea.

In [12]:
# Esta es una forma mas elegante de enviar el IF statement.

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        # Extraemos el nombre de la herramienta y sus argumentos mapeados por la IA
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        #  Buscamos la función en el espacio global usando su nombre de texto
        tool = globals().get(tool_name)
        # Si la función existe, la ejecutamos expandiendo los argumentos (**arguments)
        # Si no existe, devolvemos un diccionario vacío seguro
        result = tool(**arguments) if tool else {}
        # Guardamos el resultado con la estructura y el ID único que nos pide la API
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [14]:
# Inicializamos el lector de PDFs apuntando a tu perfil de LinkedIn exportado
reader = PdfReader("me/Profile.pdf")
linkedin = ""
# Recorremos cada página del PDF y extraemos el texto para construir tu base de conocimientos
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text
# Abrimos y leemos tu resumen profesional personalizado garantizando los acentos (utf-8)
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

# Definimos el nombre para que el sistema sepa a quién está representando
name = "Jose Manuel"

En esta celda estoy definiendo el system_prompt, es decir, 
la personalidad, las reglas de comportamiento y los límites de tu clon de IA.

Aquí le estoy inyectando directamente mi nombre, mi resumen profesional y mi LinkedIn, 
pero además le estoy dando instrucciones explícitas de cómo y cuándo usar las dos herramientas
que acabamos de crear en JSON (registrar correos de interesados y registrar preguntas desconocidas).

Opcional: Como el modelo es inteligente, ya sabe cuándo debe activarlas solo con leer el JSON.

 system_prompt por "refuerzo", A los modelos pequeños a veces les cuesta seguir las reglas de los JSONs a la primera.

In [15]:
# Definimos las instrucciones de comportamiento y personalidad de mi clon
system_prompt = f"Actúas como {name}. Estás respondiendo preguntas en el sitio web de {name}, " \
                f"particularmente preguntas relacionadas con la carrera, antecedentes, habilidades y experiencia de {name}. " \
                f"Tu responsabilidad es representar a {name} en las interacciones del sitio web de la manera más fiel posible. " \
                f"Se te proporciona un resumen de la trayectoria de {name} y su perfil de LinkedIn que puedes utilizar para responder preguntas. " \
                f"Sé profesional y carismático, como si estuvieras hablando con un cliente potencial o un futuro empleador que se topó con el sitio web. " \
                f"Si no sabes la respuesta a alguna pregunta, utiliza tu herramienta 'record_unknown_question' para registrar la pregunta que no pudiste responder, " \
                f"incluso si es sobre algo trivial o no relacionado con la carrera. " \
                f"Si el usuario entabla una conversación, intenta orientarlo para que se ponga en contacto por correo electrónico; " \
                f"pídele su correo electrónico y regístralo utilizando tu herramienta 'record_user_details'."

# Inyectamos mi base de conocimiento (el resumen de texto y el perfil de LinkedIn extraído)
system_prompt += f"\n\n## Resumen:\n{summary}\n\n## Perfil de LinkedIn:\n{linkedin}\n\n"

# Le damos la orden final para que nunca se salga del personaje de "Jose Manuel"
system_prompt += f"Con este contexto, por favor conversa con el usuario, manteniéndote siempre en el personaje de {name}."


In [ ]:
def chat(message, history):

    # --- BLOQUE DE LIMPIEZA ANTIE-RROR 400 ---
    # Limpiamos el historial de Gradio eliminando metadatos ocultos que Groq rechaza
    clean_history = []
    for h in history:
        clean_history.append({
            "role": h["role"],
            "content": h["content"]
        })

    #  Construimos el historial de la conversación inyectando tu system_prompt, 
    # el historial acumulado y el mensaje actual del usuario.
    messages = [{"role": "system", "content": system_prompt}] + clean_history + [{"role": "user", "content": message}]
    
    done = False
    while not done:
        #  Llamada al LLM. OJO: Cambiamos "gpt-4o-mini" por el modelo de Groq que estás usando
        response = openai.chat.completions.create(
            model="llama-3.3-70b-versatile",  # Pon aquí tu modelo de Groq (o el de Ollama si cambias de celda)
            messages=messages, 
            tools=tools  # Aquí le pasamos tu "cinturón" de herramientas JSON
        )

        # 3. Revisamos el motivo por el cual el modelo terminó de procesar
        finish_reason = response.choices[0].finish_reason
        
        #  Verificamos si quiere usar herramientas
        if finish_reason == "tool_calls":
            message_obj = response.choices[0].message
            tool_calls = message_obj.tool_calls
            
            # Ejecutamos las funciones dinámicamente y obtenemos los resultados
            results = handle_tool_calls(tool_calls)
            
            # Añadimos la orden del modelo y las respuestas de las herramientas al historial interno
            # para que el modelo sepa qué pasó y pueda formular su respuesta final basada en el resultado
            messages.append(message_obj) # respuesta del model
            messages.extend(results)     # respuesta de herramientas
        else:
            # Si el finish_reason es "stop", significa que el modelo ya respondió con texto normal y terminamos el bucle
            done = True
            
    # 5. Devolvemos la respuesta final en texto que leerá el usuario
    return response.choices[0].message.content

In [ ]:
# Lanzamos la interfaz de chat interactiva usando Gradio.
# Pasamos nuestra función 'chat' como el motor y definimos el formato moderno de mensajes

gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Tool called: record_user_details
Push: Recording interest from  with email  and notes Conversación sobre música y tecnología


#### Y ahora, el despliegue
Este código se encuentra en app.py.

Haremos el despliegue en HuggingFace Spaces.

Antes de empezar: ¡recuerda actualizar los archivos en el directorio "me" —tu perfil de LinkedIn y tu summary.txt— para que la IA hable de ti! Cambia también self.name = "Ed Donner" por tu nombre en app.py.

Comprueba también que no haya ningún archivo README dentro del directorio 1_foundations. 
Si hay uno, por favor elimínalo. El proceso de despliegue creará un nuevo archivo README en este directorio por ti.

Y una cosa más: esto es opcional, pero quizás quieras eliminar la carpeta completa "community_contributions" que está dentro de 1_foundations. Siempre podrás volver a descargarla de GitHub en el futuro. Si no la borras, toda esa carpeta se subirá a HuggingFace aunque no la necesitemos, y se ha vuelto bastante pesada.

Pasos para el Despliegue:
1 Visita https://huggingface.co y créate una cuenta.

2 En el menú de tu Avatar (arriba a la derecha), elige Access Tokens (Tokens de Acceso). Elige "Create New Token" (Crear nuevo token). Dale permisos de ESCRITURA (WRITE) —¡es fundamental que tenga permisos de WRITE!—. Guarda bien tu nueva clave.

3 En la Terminal, ejecuta: uv tool install 'huggingface_hub[cli]' para instalar la herramienta de HuggingFace. Luego ejecuta hf auth login --token TU_TOKEN_AQUÍ (por ejemplo, hf auth login --token hf_xxxxxx) para iniciar sesión en la línea de comandos con tu clave. Después, ejecuta hf auth whoami para verificar que estás logueado correctamente.

4 Toma tu nuevo token y añádelo a tu archivo .env como HF_TOKEN=hf_xxx para el futuro.

5 Desde la carpeta 1_foundations, introduce: uv run gradio deploy.

6 Sigue las instrucciones que aparecerán en la terminal:

    -Nombra la aplicación como "career_conversation".

    -Especifica que el archivo principal es app.py.

    -Elige cpu-basic como hardware.

    -Responde Yes (Sí) a la pregunta de si necesitas proporcionar "secrets" (variables ocultas de entorno).

    -Proporciona tu API key de OpenAI (en tu caso, la de Groq si vas a usar Groq en producción), tu usuario de Pushover y tu token de Pushover.

    -Responde No a la opción de GitHub Actions.

Gracias a Robert, James, Martins, Andras y Priya por estos consejos. > Por favor, lee las siguientes dos secciones para saber cómo cambiar tus Secrets y cómo volver a desplegar tu Space (es posible que tengas que borrar el archivo README.md que se creará en este directorio 1_foundations).

Más información sobre los "Secrets" (Variables Secretas):
Si te confunde lo que está pasando con los secrets: el sistema simplemente quiere que introduzcas el nombre de la variable y su valor para cada uno de tus secretos. Por lo tanto, tendrías que escribir:

OPENAI_API_KEY

Seguido de (pulsando Enter):

sk-proj-...

Si no quieres configurar los secretos de esta manera, o algo sale mal, no hay problema; puedes cambiar tus secretos más tarde desde la web:

Inicia sesión en el sitio web de HuggingFace.

Ve a la pantalla de tu perfil a través del menú del Avatar (arriba a la derecha).

Selecciona el Space que acabas de desplegar.

Haz clic en la rueda dentada de Settings (Configuración) arriba a la derecha.

Desplázate hacia abajo hasta la sección Variables and Secrets para cambiar tus claves, borrar el Space, etc.

¡Y ahora ya deberías estar desplegado!
Si en algún momento quieres reemplazar por completo todo el despliegue y empezar de nuevo desde cero con tus claves, es posible que tengas que borrar el archivo README.md que se creó automáticamente en esta carpeta 1_foundations.

Aquí puedes ver el mío: https://huggingface.co/spaces/ed-donner/Career_Conversation

Acabo de recibir una notificación push de un estudiante preguntándome cómo puede convertirse en presidente de su país 😂😂

Para más información sobre el despliegue:

https://www.gradio.app/guides/sharing-your-app#hosting-on-hf-spaces

Para eliminar tu Space en el futuro:
Inicia sesión en HuggingFace.

En el menú del Avatar, selecciona tu perfil.

Haz clic en el Space y selecciona la rueda de configuración (Settings) arriba a la derecha.

Desplázate hasta la sección Delete (Eliminar) abajo del todo.

ADEMÁS: elimina el archivo README que Gradio habrá creado dentro de esta carpeta 1_foundations (de lo contrario, no te volverá a hacer las preguntas de configuración la próxima vez que ejecutes uv run gradio deploy).

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Ejercicio</h2>
             <span style="color:#ff7800;">• Primero y ante todo, ¡despliega esto por tu cuenta! Es una herramienta real y valiosa: el futuro currículum.<br/>
            • A continuación, mejora los recursos: añade mejor contexto sobre ti mismo. Si sabes RAG, agrega una base de conocimientos sobre ti.<br/>
            • ¡Añade más herramientas! ¿Podrías tener una base de datos SQL con preguntas y respuestas comunes que el LLM pudiera leer y escribir?<br/>
            • Incorpora el Evaluador del laboratorio anterior y añade otros patrones de agentes.
            . que la aplacion se vea mas bonita, streaming de resultados
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Implicaciones Comerciales</h2>
           <span style="color:#00bfff;">Aparte de lo obvio (tu alter ego profesional), esto tiene aplicaciones empresariales en cualquier situación donde necesites un asistente de IA con experiencia en un dominio específico y capacidad para interactuar con el mundo real.
        </span>
        </td>
    </tr>
</table>